# App-12 : Puissance 4 — IA de jeu et comparaison d'algorithmes

**Navigation** : [<< App-11 Picross](App-11-Picross-Python.ipynb) | [Index](../README.md) | [App-13 TSP Metaheuristiques >>](App-13-TSP-Metaheuristics-Python.ipynb)

## Objectifs d'apprentissage

Ce carnet est le jumeau **Rust** de [App-12 ConnectFour](App-12-ConnectFour-Python.ipynb).
Le jeu, les algorithmes et les positions de demonstration sont les memes ; la
forme change la ou Python s'appuyait sur une bibliotheque.

A la fin, on sait :

- representer un plateau de Puissance 4 et sa logique de victoire ;
- ecrire une IA aleatoire, une heuristique gloutonne, un **minimax alpha-beta**
  et un **MCTS** avec UCB1 ;
- confronter ces IA dans un tournoi round-robin et lire un classement ;
- reconnaitre le cadre **AIMA** (etats, actions, test terminal, utilite) derriere
  une implementation concrete ;
- distinguer les techniques d'elagage et d'ordonnancement qui font la difference
  en pratique.

## L'organe : ce que Rust std fournit, ce qu'il ne fournit pas

Le jumeau Python s'appuie sur trois bibliotheques : `numpy` pour les moyennes,
`matplotlib` pour les plateaux et les graphiques, et le module `random` pour le
hasard. Aucune n'existe dans la bibliotheque standard de Rust. Le carnet ne les
simule pas : il les **remplace explicitement**, et chaque substitution est
declaree ici.

| Besoin du jumeau Python | Substitut Rust | Pourquoi c'est equivalent |
|---|---|---|
| `matplotlib` : plateau en couleurs | `display_board` en texte, avec titre et indicateur de colonne | la position et le coup joue restent lisibles ; les couleurs ne portent aucune information de jeu |
| `matplotlib` : graphiques d'evolution | lignes de donnees imprimees, une par graphique | les valeurs affichees sont les memes ; le dessin disparait, les nombres restent |
| `numpy.mean` | `mean` sur `&[f64]` | meme moyenne arithmetique |
| durees : `{:.3f} s`, `{:.1f} ms` | millisecondes a trois decimales (`{:.3f} ms`) | meme mesure, a l'echelle de Rust : une decision sous la milliseconde s'afficherait `0.000` en secondes |
| `random.seed(int)` + `random.choice` | `Mt19937`, generateur de la bibliotheque standard de CPython | les suites de coups sont **identiques** au jumeau Python |

Le generateur merite un mot. Reproduire `random.seed(42)` de CPython ne
consiste pas a prendre un generateur « au hasard » : il faut refaire
**MT19937** avec son amorcage `init_by_array`, sa torsion modulaire et son
trempage, puis `getrandbits` et `randbelow` avec le **rejet** des valeurs hors
borne. C'est ce qui permet de retrouver, plus bas, les memes colonnes jouees
que le carnet Python.

## Le socle : constantes et generateur

On commence par les constantes du jeu et par le generateur. Le plateau lui-meme
vient ensuite : une structure `Copy`, dont l'affectation suffit a produire une
copie independante — ce que Python obtenait par `grid.copy()`, appele dans
`Board.copy()`.

In [2]:
// Constantes du jeu
#[allow(dead_code)] const ROWS: usize = 6;
#[allow(dead_code)] const COLS: usize = 7;
#[allow(dead_code)] const EMPTY: i32 = 0;
#[allow(dead_code)] const PLAYER1: i32 = 1;
#[allow(dead_code)] const PLAYER2: i32 = 2;
#[allow(dead_code)] const CONNECT: usize = 4;

#[allow(dead_code)]
fn other(player: i32) -> i32 {
    if player == PLAYER1 { PLAYER2 } else { PLAYER1 }
}

/// MT19937 reproduisant exactement `random.seed(int)` de CPython.
///
/// Deux ecarts avec le generateur de reference, tous deux necessaires a
/// l'egalite des tirages :
/// - l'amorcage passe par `init_by_array` sur les mots de 32 bits du germe,
///   puis `mt[0] = 0x80000000` ;
/// - `randbelow` tire `getrandbits(64 - n.leading_zeros())` et **rejette**
///   les valeurs hors borne, meme pour n = 1 : CPython consomme au moins un
///   bit a chaque appel, l'omettre decalerait toute la suite.
#[allow(dead_code)]
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32
                .wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30))
                .wrapping_add(i as u32);
        }
        mt
    }

    fn new(seed: u32) -> Mt19937 {
        Self::new_by_array(&[seed])
    }

    fn new_by_array(key: &[u32]) -> Mt19937 {
        let mut mt = Self::init_genrand(19650218);
        let mut i = 1usize;
        let mut j = 0usize;
        let rounds = if 624 > key.len() { 624 } else { key.len() };
        for _ in 0..rounds {
            mt[i] = (mt[i] ^ ((mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525)))
                .wrapping_add(key[j])
                .wrapping_add(j as u32);
            i += 1;
            j += 1;
            if i >= 624 {
                mt[0] = mt[623];
                i = 1;
            }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ ((mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941)))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 {
                mt[0] = mt[623];
                i = 1;
            }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }

    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for kk in 0..624 {
                let y = (self.mt[kk] & 0x8000_0000) | (self.mt[(kk + 1) % 624] & 0x7fff_ffff);
                let mag = if y & 1 != 0 { 0x9908_b0df } else { 0 };
                self.mt[kk] = self.mt[(kk + 397) % 624] ^ (y >> 1) ^ mag;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }

    fn getrandbits(&mut self, k: u32) -> u64 {
        if k == 0 { return 0; }
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }

    fn randbelow(&mut self, n: u64) -> u64 {
        // Python leve une exception sur n = 0 ; ici un zero neutre evite la
        // boucle infinie que produirait `k = 0` (`while 0 >= 0`).
        if n == 0 { return 0; }
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n {
            r = self.getrandbits(k);
        }
        r
    }

    /// Equivalent de `random.choice(seq)` : un tirage uniforme par indice.
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T {
        let i = self.randbelow(seq.len() as u64) as usize;
        &seq[i]
    }
}

println!("Puissance 4 : plateau {}x{}, alignement de {}", ROWS, COLS, CONNECT);
println!("Environnement pret.");

Puissance 4 : plateau 6x7, alignement de 4


Environnement pret.


### Lecture du resultat

La premiere ligne confirme les dimensions lues dans les constantes, et
`Environnement pret.` signale que le generateur est construit. Aucune sortie
supplementaire n'est attendue : sous `evcxr`, une cellule de declarations
n'affiche que ses `println!`.

## Le plateau

`Board` porte une grille `[[i32; COLS]; ROWS]` et les operations du jeu :
coups legaux, chute d'un jeton, retrait, detection de victoire dans les quatre
directions, test terminal et comptage des jetons. Le rendu texte remplace
`__repr__`.

In [3]:
/// Plateau de Puissance 4 avec logique de jeu complete.
///
/// `Copy` remplace `Board.copy()` du jumeau Python : une affectation suffit a
/// obtenir un plateau independant, ce qui rend `undo_move` inutile pour la copie.
#[derive(Clone, Copy)]
#[allow(dead_code)]
struct Board { grid: [[i32; COLS]; ROWS] }

#[allow(dead_code)]
impl Board {
    fn new() -> Board {
        Board { grid: [[EMPTY; COLS]; ROWS] }
    }

    /// Retourne la liste des colonnes jouables (non pleines).
    fn legal_moves(&self) -> Vec<usize> {
        (0..COLS).filter(|&c| self.grid[0][c] == EMPTY).collect()
    }

    /// Place un jeton dans la colonne donnee. Retourne la ligne d'atterrissage.
    fn drop_piece(&mut self, col: usize, player: i32) -> i32 {
        for row in (0..ROWS).rev() {
            if self.grid[row][col] == EMPTY {
                self.grid[row][col] = player;
                return row as i32;
            }
        }
        -1
    }

    /// Retire le jeton du haut de la colonne donnee.
    fn undo_move(&mut self, col: usize) {
        for row in 0..ROWS {
            if self.grid[row][col] != EMPTY {
                self.grid[row][col] = EMPTY;
                return;
            }
        }
    }

    /// Verifie si le joueur a aligne CONNECT jetons.
    fn check_win(&self, player: i32) -> bool {
        let g = &self.grid;
        // Horizontal
        for r in 0..ROWS {
            for c in 0..COLS - 3 {
                if (0..CONNECT).all(|i| g[r][c + i] == player) { return true; }
            }
        }
        // Vertical
        for r in 0..ROWS - 3 {
            for c in 0..COLS {
                if (0..CONNECT).all(|i| g[r + i][c] == player) { return true; }
            }
        }
        // Diagonale montante
        for r in 3..ROWS {
            for c in 0..COLS - 3 {
                if (0..CONNECT).all(|i| g[r - i][c + i] == player) { return true; }
            }
        }
        // Diagonale descendante
        for r in 0..ROWS - 3 {
            for c in 0..COLS - 3 {
                if (0..CONNECT).all(|i| g[r + i][c + i] == player) { return true; }
            }
        }
        false
    }

    /// `None` tant que la partie continue, `Some(gagnant)` sinon (0 = nul).
    fn is_terminal(&self) -> Option<i32> {
        if self.check_win(PLAYER1) { return Some(PLAYER1); }
        if self.check_win(PLAYER2) { return Some(PLAYER2); }
        if self.legal_moves().is_empty() { return Some(0); }
        None
    }

    /// Nombre de jetons poses sur le plateau.
    fn move_count(&self) -> usize {
        self.grid.iter().flatten().filter(|&&v| v != EMPTY).count()
    }

    /// Rendu texte, equivalent de `__repr__`.
    fn render(&self) -> String {
        let mut out = String::new();
        for r in 0..ROWS {
            for c in 0..COLS {
                if c > 0 { out.push(' '); }
                out.push(match self.grid[r][c] {
                    PLAYER1 => 'R',
                    PLAYER2 => 'J',
                    _ => '.',
                });
            }
            out.push('\n');
        }
        let header: Vec<String> = (0..COLS).map(|c| c.to_string()).collect();
        out.push_str(&header.join(" "));
        out
    }
}

/// Rendu d'une liste d'indices au format Python : `[0, 1, 2]`.
#[allow(dead_code)]
fn py_list(v: &[usize]) -> String {
    let parts: Vec<String> = v.iter().map(|x| x.to_string()).collect();
    format!("[{}]", parts.join(", "))
}

/// Moyenne d'une serie de durees (`0.0` sur une serie vide).
#[allow(dead_code)]
fn mean(v: &[f64]) -> f64 {
    if v.is_empty() { 0.0 } else { v.iter().sum::<f64>() / v.len() as f64 }
}

let mut board = Board::new();
board.drop_piece(3, PLAYER1);
board.drop_piece(3, PLAYER2);
board.drop_piece(4, PLAYER1);

println!("{}", board.render());
println!("\nCoups legaux : {}", py_list(&board.legal_moves()));
println!("Jetons poses : {}", board.move_count());

. . . . . . .


. . . . . . .


. . . . . . .


. . . . . . .


. . . J . . .


. . . R R . .


0 1 2 3 4 5 6


Coups legaux : [0, 1, 2, 3, 4, 5, 6]


Jetons poses : 3


### Lecture du resultat

La colonne 3 porte deux jetons empiles (Rouge en bas, Jaune au-dessus) et la
colonne 4 un jeton Rouge : la ligne du bas se lit donc `. . . R R . .`, et
l'avant-derniere porte le Jaune seul en colonne 3. Les sept colonnes restent
jouables et le plateau compte trois jetons — la coherence entre `legal_moves` et
`move_count` est verifiee du meme geste.

## Affichage du plateau

Le jumeau Python dessine un plateau bleu avec des cercles rouges et jaunes.
Faute de bibliotheque graphique dans la bibliotheque standard, la version Rust
affiche le meme contenu en texte : titre, grille, numeros de colonnes, et un
`v` sous la colonne mise en avant. L'information pedagogique — ou en est la
partie — est conservee ; seule la mise en forme change.

In [4]:
/// Rendu texte du plateau, substitut de la figure matplotlib du jumeau Python.
///
/// Le titre, la grille et l'indicateur de colonne sont conserves ; les couleurs
/// et les cercles ne le sont pas, faute de bibliotheque graphique en Rust std.
#[allow(dead_code)]
fn display_board(board: &Board, title: &str, highlight_col: Option<usize>) {
    println!("{}", title);
    println!("{}", board.render());
    // L'indicateur ne s'imprime que si une colonne est mise en avant, comme la
    // fleche de la figure Python : un appel sans surlignage n'ajoute rien.
    if let Some(col) = highlight_col {
        let marks: Vec<&str> = (0..COLS)
            .map(|c| if c == col { "v" } else { " " })
            .collect();
        println!("{}", marks.join(" "));
    }
}

// Demonstration : une partie en cours
let mut demo_board = Board::new();
let moves = [(3usize, PLAYER1), (3, PLAYER2), (4, PLAYER1), (2, PLAYER2),
             (5, PLAYER1), (4, PLAYER2), (4, PLAYER1), (5, PLAYER2)];
for (col, player) in moves {
    demo_board.drop_piece(col, player);
}

display_board(&demo_board, "Partie en cours (8 coups joues)", None);

Partie en cours (8 coups joues)


. . . . . . .


. . . . . . .


. . . . . . .


. . . . R . .


. . . J J J .


. . J R R R .


0 1 2 3 4 5 6


### Interpretation

Cette position de huit coups sert de poste d'observation stable : les cellules
suivantes y appliquent les fonctions d'evaluation, ce qui rend leurs scores
comparables d'une cellule a l'autre. Elle est construite une seule fois, puis
reutilisee — le binding `demo_board` persiste d'une cellule a l'autre sous
`evcxr`.

## La boucle de partie

`play_game` fait s'affronter deux IA et retourne le gagnant, l'historique des
coups et les durees par joueur. En Python, les IA lisaient le module `random`
global ; ici le generateur est **passe en parametre**, ce qui rend la suite de
coups reproductible sans etat cache.

In [5]:
use std::time::Instant;

/// Fait jouer `ai1` (joueur 1) contre `ai2` (joueur 2).
///
/// Chaque IA est une fonction `(plateau, joueur, generateur) -> colonne`.
/// Le generateur est passe explicitement la ou Python lit le module `random` :
/// c'est ce fil unique qui rend les suites reproductibles.
/// `verbose` imprime chaque coup. Le jumeau Python affiche aussi une figure de
/// fin de partie ; la spec du lot la declare absente cote Rust, la sortie texte
/// (coups et resultat) etant le seul temoin. Le drapeau `_display` garde donc la
/// parite de signature sans rien imprimer.
/// Retourne `(gagnant, historique des coups, durees par joueur)`.
#[allow(dead_code)]
fn play_game(
    ai1: &dyn Fn(&Board, i32, &mut Mt19937) -> usize,
    ai2: &dyn Fn(&Board, i32, &mut Mt19937) -> usize,
    rng: &mut Mt19937,
    verbose: bool,
    _display: bool,
) -> (i32, Vec<(usize, i32)>, [Vec<f64>; 2]) {
    let mut board = Board::new();
    let mut current = PLAYER1;
    let mut history: Vec<(usize, i32)> = Vec::new();
    let mut times: [Vec<f64>; 2] = [Vec::new(), Vec::new()];

    loop {
        let t0 = Instant::now();
        let col = if current == PLAYER1 {
            ai1(&board, current, rng)
        } else {
            ai2(&board, current, rng)
        };
        let elapsed = t0.elapsed().as_secs_f64();
        times[(current - 1) as usize].push(elapsed);

        board.drop_piece(col, current);
        history.push((col, current));

        if verbose {
            let player_name = if current == PLAYER1 { "Rouge" } else { "Jaune" };
            println!("Coup {:2}: {} joue colonne {} ({:.3} ms)", history.len(), player_name, col, elapsed * 1000.0);
        }

        if let Some(winner) = board.is_terminal() {
            return (winner, history, times);
        }

        current = other(current);
    }
}

println!("Fonction play_game definie.");

Fonction play_game definie.


## IA aleatoire

`ai_random` choisit uniformement parmi les coups legaux. C'est la reference
basse du carnet : toute IA qui ne la bat pas decisivement a un probleme.

La graine vaut 42, comme dans le jumeau Python ; le generateur reproduit
CPython, donc les colonnes jouees doivent etre **identiques**.

In [6]:
/// IA aleatoire : choisit une colonne legale au hasard.
#[allow(dead_code)]
fn ai_random(board: &Board, _player: i32, rng: &mut Mt19937) -> usize {
    let moves = board.legal_moves();
    *rng.choice(&moves)
}

// Demonstration : Random vs Random
let mut rng = Mt19937::new(42);
let (winner, history, _times) = play_game(&ai_random, &ai_random, &mut rng, true, true);

let result = if winner == 0 {
    "Match nul"
} else if winner == PLAYER1 {
    "Rouge gagne"
} else {
    "Jaune gagne"
};
println!("\nResultat : {} en {} coups", result, history.len());

Coup  1: Rouge joue colonne 5 (0.004 ms)


Coup  2: Jaune joue colonne 0 (0.000 ms)


Coup  3: Rouge joue colonne 0 (0.000 ms)


Coup  4: Jaune joue colonne 5 (0.000 ms)


Coup  5: Rouge joue colonne 2 (0.000 ms)


Coup  6: Jaune joue colonne 1 (0.000 ms)


... (11 outputs left out)

Coup 18: Jaune joue colonne 0 (0.000 ms)


Coup 19: Rouge joue colonne 1 (0.000 ms)


Coup 20: Jaune joue colonne 2 (0.000 ms)


Resultat : Jaune gagne en 20 coups


### Lecture du resultat

Chaque ligne `Coup N` donne le joueur, la colonne et la duree de la decision.
Les deux joueurs tirent dans le **meme** flux de nombres : c'est exactement ce
que faisait `random.seed(42)` suivi de deux appels a `ai_random`, et c'est ce
qui rend la colonne jouee a chaque tour verifiable contre le carnet Python.

## Evaluation heuristique

Une recherche ne peut pas descendre jusqu'a la fin de toutes les parties : il
lui faut une note de position. `evaluate_window` note une fenetre de quatre
cases (quatre pions valent beaucoup, trois pions et une case vide un peu), et
`evaluate_position` somme ces fenetres sur les quatre orientations, avec un
bonus pour la colonne centrale.

Les poids ne sont pas arbitraires, mais ils ne sont pas symetriques : une
menace propre rapporte un peu plus qu'une menace adverse ne coute, et le bonus
de colonne centrale ne compte que pour le joueur evalue. Les deux notes d'une
meme position ne s'opposent donc pas exactement : c'est leur ecart, affiche en
fin de sortie, qui mesure l'avantage.

In [7]:
/// Evalue une fenetre de 4 cases pour le joueur donne.
#[allow(dead_code)]
fn evaluate_window(window: &[i32; CONNECT], player: i32) -> i32 {
    let opponent = other(player);
    let player_count = window.iter().filter(|&&v| v == player).count();
    let opponent_count = window.iter().filter(|&&v| v == opponent).count();
    let empty_count = window.iter().filter(|&&v| v == EMPTY).count();
    let mut score = 0;

    if player_count == 4 {
        score += 1000;
    } else if player_count == 3 && empty_count == 1 {
        score += 10;
    } else if player_count == 2 && empty_count == 2 {
        score += 2;
    }

    if opponent_count == 3 && empty_count == 1 {
        score -= 8;
    }

    score
}

/// Evalue la position globale du plateau pour le joueur donne.
#[allow(dead_code)]
fn evaluate_position(board: &Board, player: i32) -> i32 {
    let mut score = 0;
    let g = &board.grid;

    // Bonus pour les jetons au centre (colonne 3)
    score += g.iter().filter(|row| row[COLS / 2] == player).count() as i32 * 3;

    // Horizontal
    for r in 0..ROWS {
        for c in 0..COLS - 3 {
            score += evaluate_window(&[g[r][c], g[r][c + 1], g[r][c + 2], g[r][c + 3]], player);
        }
    }
    // Vertical
    for r in 0..ROWS - 3 {
        for c in 0..COLS {
            score += evaluate_window(&[g[r][c], g[r + 1][c], g[r + 2][c], g[r + 3][c]], player);
        }
    }
    // Diagonale descendante
    for r in 0..ROWS - 3 {
        for c in 0..COLS - 3 {
            score += evaluate_window(
                &[g[r][c], g[r + 1][c + 1], g[r + 2][c + 2], g[r + 3][c + 3]],
                player,
            );
        }
    }
    // Diagonale montante
    for r in 3..ROWS {
        for c in 0..COLS - 3 {
            score += evaluate_window(
                &[g[r][c], g[r - 1][c + 1], g[r - 2][c + 2], g[r - 3][c + 3]],
                player,
            );
        }
    }

    score
}

let score_p1 = evaluate_position(&demo_board, PLAYER1);
let score_p2 = evaluate_position(&demo_board, PLAYER2);
println!("Score joueur 1 (Rouge) : {}", score_p1);
println!("Score joueur 2 (Jaune) : {}", score_p2);
println!(
    "Avantage : {} (+{})",
    if score_p1 > score_p2 { "Rouge" } else { "Jaune" },
    (score_p1 - score_p2).abs()
);

Score joueur 1 (Rouge) : -3


Score joueur 2 (Jaune) : 17


Avantage : Jaune (+20)


### Interpretation

Les deux scores ne sont pas opposes : chaque joueur est evalue avec **ses**
pions. L'ecart mesure donc l'avantage de position, et son signe designe le
camp qui mene. Sur la position de demonstration, l'avantage va a Jaune : les
jetons jaunes y occupent des fenetres plus prometteuses que les rouges.

## Minimax alpha-beta

`minimax` explore l'arbre des coups en alternant maximisation et minimisation.
Deux details font la difference : les etats terminaux sont notes avec un bonus
de **profondeur** (une victoire proche vaut mieux qu'une victoire lointaine) et
les coups sont essayes du centre vers les bords, ce qui fait couper plus tot.

`make_minimax_ai(depth)` emballe la recherche dans une fonction du type attendu
par `play_game`.

In [8]:
/// Minimax avec elagage alpha-beta.
///
/// Le plateau est modifie en place puis restaure par `undo_move`, comme en
/// Python : aucun plateau intermediaire n'est alloue pendant la recherche.
/// Retourne `(colonne, score)`.
#[allow(dead_code)]
fn minimax(
    board: &mut Board,
    depth: i32,
    mut alpha: f64,
    mut beta: f64,
    maximizing: bool,
    player: i32,
) -> (Option<usize>, f64) {
    let opponent = other(player);

    // Cas terminaux
    if let Some(winner) = board.is_terminal() {
        if winner == player {
            return (None, 100000.0 + depth as f64); // Victoire (preferer les victoires rapides)
        } else if winner == opponent {
            return (None, -100000.0 - depth as f64); // Defaite
        } else {
            return (None, 0.0); // Match nul
        }
    }

    if depth == 0 {
        return (None, evaluate_position(board, player) as f64);
    }

    // Ordre d'exploration : centre d'abord (heuristique d'ordonnancement)
    let mut moves = board.legal_moves();
    moves.sort_by_key(|&c| ((c as i32) - (COLS / 2) as i32).abs());
    let first = moves.first().copied().unwrap_or(0);

    if maximizing {
        let mut best_score = f64::NEG_INFINITY;
        let mut best_col = first;
        for col in moves {
            board.drop_piece(col, player);
            let (_c, score) = minimax(board, depth - 1, alpha, beta, false, player);
            board.undo_move(col);
            if score > best_score {
                best_score = score;
                best_col = col;
            }
            if best_score > alpha { alpha = best_score; }
            if alpha >= beta {
                break; // Coupure beta
            }
        }
        (Some(best_col), best_score)
    } else {
        let mut best_score = f64::INFINITY;
        let mut best_col = first;
        for col in moves {
            board.drop_piece(col, opponent);
            let (_c, score) = minimax(board, depth - 1, alpha, beta, true, player);
            board.undo_move(col);
            if score < best_score {
                best_score = score;
                best_col = col;
            }
            if best_score < beta { beta = best_score; }
            if alpha >= beta {
                break; // Coupure alpha
            }
        }
        (Some(best_col), best_score)
    }
}

/// Cree une IA Minimax avec la profondeur donnee.
#[allow(dead_code)]
fn make_minimax_ai(depth: i32) -> impl Fn(&Board, i32, &mut Mt19937) -> usize {
    move |board: &Board, player: i32, _rng: &mut Mt19937| -> usize {
        let mut work = *board;
        let (col, _score) = minimax(&mut work, depth, f64::NEG_INFINITY, f64::INFINITY, true, player);
        match col { Some(c) => c, None => 0 }
    }
}

println!("Minimax alpha-beta implemente.");
println!("Profondeurs recommandees : 2 (rapide), 4 (bon), 6 (fort mais lent)");

Minimax alpha-beta implemente.


Profondeurs recommandees : 2 (rapide), 4 (bon), 6 (fort mais lent)


## Minimax en profondeur 4 contre l'IA aleatoire

Premier test de force : une recherche serieuse doit battre le hasard, et
rapidement.

In [9]:
// Minimax (profondeur 4) vs Random
// evcxr ne sait pas persister un `impl Fn` : on l'emballe dans un objet trait.
let ai_mm4: Box<dyn Fn(&Board, i32, &mut Mt19937) -> usize> = Box::new(make_minimax_ai(4));

let mut rng = Mt19937::new(123);
let (winner, history, times) = play_game(&ai_mm4, &ai_random, &mut rng, true, true);

let result_str = if winner == 0 {
    "Match nul"
} else if winner == PLAYER1 {
    "Minimax gagne"
} else {
    "Random gagne"
};
println!("\nResultat : {} en {} coups", result_str, history.len());
println!("Temps moyen Minimax : {:.3} ms/coup", mean(&times[0]) * 1000.0);
println!("Temps moyen Random  : {:.3} ms/coup", mean(&times[1]) * 1000.0);


Coup  1: Rouge joue colonne 3 (0.155 ms)


Coup  2: Jaune joue colonne 0 (0.002 ms)


Coup  3: Rouge joue colonne 4 (0.416 ms)


Coup  4: Jaune joue colonne 2 (0.001 ms)


Coup  5: Rouge joue colonne 3 (0.467 ms)


Coup  6: Jaune joue colonne 0 (0.000 ms)


... (2 outputs left out)

Coup  9: Rouge joue colonne 3 (0.087 ms)


Resultat : Minimax gagne en 9 coups


Temps moyen Minimax : 0.322 ms/coup


Temps moyen Random  : 0.001 ms/coup


### Lecture du resultat

La colonne jouee par chaque camp figure en clair, ce qui permet de relire la
partie coup par coup. Le contraste des temps moyens est l'information
principale : la recherche coute des ordres de grandeur de plus qu'un tirage
aleatoire, et c'est ce cout qu'il faudra contenir en augmentant la profondeur.

## Le cout de la profondeur

Trois profondeurs, dix parties chacune, contre la meme IA aleatoire. Le nombre
de victoires dit si la profondeur suffit ; le temps par coup dit ce qu'elle
coute.

In [10]:
// Comparer les profondeurs 2, 4 et 6
let depths = [2, 4, 6];
let n_games = 10;

for &d in &depths {
    let ai_mm = make_minimax_ai(d);
    let mut wins = 0usize;
    let mut total_time = 0.0;
    let mut total_moves = 0usize;

    for game_i in 0..n_games {
        let mut rng = Mt19937::new((game_i * 100) as u32);
        let (winner, _history, times) = play_game(&ai_mm, &ai_random, &mut rng, false, false);
        if winner == PLAYER1 {
            wins += 1;
        }
        total_time += times[0].iter().sum::<f64>();
        total_moves += times[0].len();
    }

    let avg_time = if total_moves > 0 { total_time / total_moves as f64 } else { 0.0 };
    println!(
        "Minimax(d={}) : {}/{} victoires, {:.3} ms/coup",
        d, wins, n_games, avg_time * 1000.0
    );
}

// Le jumeau Python trace ici deux graphiques ; en Rust std, les trois lignes
// ci-dessus portent les memes valeurs, resumees par la ligne suivante.
println!("Profondeurs comparees : d=2, d=4, d=6 ({} parties chacune)", n_games);
let _ = ();  // evcxr : la cellule se termine sur une instruction, rien a afficher


Minimax(d=2) : 10/10 victoires, 0.022 ms/coup


Minimax(d=4) : 10/10 victoires, 0.313 ms/coup


Minimax(d=6) : 10/10 victoires, 3.644 ms/coup


Profondeurs comparees : d=2, d=4, d=6 (10 parties chacune)


### Interpretation

Deux lectures se croisent ici. En **force**, la profondeur 2 suffit deja contre
un joueur aleatoire : le hasard ne construit pas de menace, et une recherche
courte voit la victoire immediate. En **cout**, chaque niveau supplementaire
coute plus cher, mais bien moins vite que ne le voudrait un facteur egal au
nombre de coups legaux : l'elagage alpha-beta en coupe l'essentiel, et c'est cet
elagage qu'un meilleur ordre des coups ameliore encore.

## MCTS : chercher autrement

Le minimax suppose un adversaire parfait et une fonction d'evaluation fiable.
MCTS prend le probleme par l'autre bout : il joue des parties au hasard depuis
la position courante et retient le coup dont les simulations ont le mieux
tourne. UCB1 arbitre entre exploiter un coup prometteur et explorer un coup peu
essaye.

In [11]:
/// Noeud de l'arbre MCTS, stocke dans un arbre tabulaire.
///
/// Un `Vec<MctsNode>` remplace les references `parent`/`children` de Python :
/// les liens sont des indices, ce qui evite `Rc<RefCell<...>>` et permet de
/// modifier les statistiques pendant la retropropagation sans emprunt conflictuel.
#[allow(dead_code)]
struct MctsNode {
    board: Board,
    player: i32,
    parent: Option<usize>,
    children: Vec<usize>,
    untried: Vec<usize>,
    move_played: Option<usize>,
    visits: u32,
    wins: f64,
}

#[allow(dead_code)]
impl MctsNode {
    fn new(board: Board, player: i32, parent: Option<usize>, move_played: Option<usize>) -> MctsNode {
        let untried = board.legal_moves();
        MctsNode { board, player, parent, children: Vec::new(), untried, move_played, visits: 0, wins: 0.0 }
    }

    fn is_fully_expanded(&self) -> bool {
        self.untried.is_empty()
    }
}

/// Score UCB1 d'un noeud, calcule depuis l'arbre (le parent porte `log(N)`).
#[allow(dead_code)]
fn ucb1(nodes: &[MctsNode], idx: usize, exploration: f64) -> f64 {
    let node = &nodes[idx];
    if node.visits == 0 {
        return f64::INFINITY;
    }
    let parent_visits = match node.parent {
        Some(p) => nodes[p].visits.max(1) as f64,
        None => node.visits as f64,
    };
    let exploitation = node.wins / node.visits as f64;
    let exploration_term = exploration * (parent_visits.ln() / node.visits as f64).sqrt();
    exploitation + exploration_term
}

/// Enfant de meilleur score UCB1 (premier maximum strict, comme `max(key=)`).
#[allow(dead_code)]
fn best_child(nodes: &[MctsNode], idx: usize, exploration: f64) -> usize {
    let mut best = nodes[idx].children[0];
    let mut best_score = ucb1(nodes, best, exploration);
    for &c in &nodes[idx].children[1..] {
        let score = ucb1(nodes, c, exploration);
        if score > best_score {
            best_score = score;
            best = c;
        }
    }
    best
}

/// Recherche MCTS. Retourne la colonne choisie.
#[allow(dead_code)]
fn mcts_search(
    board: &Board,
    player: i32,
    iterations: u32,
    exploration: f64,
    rng: &mut Mt19937,
) -> usize {
    // La racine represente la position vue depuis l'adversaire : son champ
    // `player` est celui qui vient de jouer, comme dans la version Python.
    let mut nodes: Vec<MctsNode> = vec![MctsNode::new(*board, other(player), None, None)];

    for _ in 0..iterations {
        // 1. Selection : descendre avec UCB1
        let mut idx = 0usize;
        while nodes[idx].is_fully_expanded() && !nodes[idx].children.is_empty() {
            idx = best_child(&nodes, idx, exploration);
        }

        // 2. Expansion : ajouter un enfant
        let terminal = nodes[idx].board.is_terminal().is_some();
        if !terminal && !nodes[idx].untried.is_empty() {
            let pick = rng.randbelow(nodes[idx].untried.len() as u64) as usize;
            let action = nodes[idx].untried.remove(pick);
            let mover = other(nodes[idx].player);
            let mut nb = nodes[idx].board;
            nb.drop_piece(action, mover);
            let child = nodes.len();
            nodes.push(MctsNode::new(nb, mover, Some(idx), Some(action)));
            nodes[idx].children.push(child);
            idx = child;
        }

        // 3. Simulation (rollout aleatoire)
        let mut rollout_board = nodes[idx].board;
        let mut rollout_player = other(nodes[idx].player);
        let winner = loop {
            if let Some(w) = rollout_board.is_terminal() {
                break w;
            }
            let moves = rollout_board.legal_moves();
            let col = *rng.choice(&moves);
            rollout_board.drop_piece(col, rollout_player);
            rollout_player = other(rollout_player);
        };

        // 4. Retropropagation
        let mut cursor = Some(idx);
        while let Some(c) = cursor {
            nodes[c].visits += 1;
            if winner == nodes[c].player {
                nodes[c].wins += 1.0;
            } else if winner == 0 {
                nodes[c].wins += 0.5; // Demi-point pour le nul
            }
            cursor = nodes[c].parent;
        }
    }

    // Choisir le coup le plus visite (pas le meilleur UCB1)
    if nodes[0].children.is_empty() {
        let moves = board.legal_moves();
        return if moves.is_empty() { 0 } else { *rng.choice(&moves) };
    }
    let mut best = nodes[0].children[0];
    let mut best_visits = nodes[best].visits;
    for &c in &nodes[0].children[1..] {
        if nodes[c].visits > best_visits {
            best_visits = nodes[c].visits;
            best = c;
        }
    }
    nodes[best].move_played.unwrap_or(0)
}

/// Cree une IA MCTS avec le nombre d'iterations donne.
#[allow(dead_code)]
fn make_mcts_ai(iterations: u32) -> impl Fn(&Board, i32, &mut Mt19937) -> usize {
    move |board: &Board, player: i32, rng: &mut Mt19937| -> usize {
        mcts_search(board, player, iterations, 1.414, rng)
    }
}

println!("MCTS implemente.");
println!("Iterations recommandees : 100 (rapide), 500 (bon), 1000 (fort), 5000 (tres fort)");

MCTS implemente.


Iterations recommandees : 100 (rapide), 500 (bon), 1000 (fort), 5000 (tres fort)


## MCTS contre Minimax

Le test qui compte : un MCTS a nombre d'iterations raisonnable contre un
minimax de profondeur 4. Aucun des deux n'est « le bon » algorithme — le
resultat depend du budget de calcul accorde a chacun.

In [12]:
// MCTS (1000 iterations) vs Minimax (profondeur 4)
// Meme raison qu'a la cellule du duel precedent : `impl Fn` n'est pas
// persistable par evcxr, donc on passe par des objets trait.
let ai_mcts_1000: Box<dyn Fn(&Board, i32, &mut Mt19937) -> usize> = Box::new(make_mcts_ai(1000));
let ai_mm4: Box<dyn Fn(&Board, i32, &mut Mt19937) -> usize> = Box::new(make_minimax_ai(4));

let mut rng = Mt19937::new(42);
let (winner, history, times) = play_game(&ai_mcts_1000, &ai_mm4, &mut rng, true, true);

let result_str = if winner == 0 {
    String::from("Match nul")
} else if winner == PLAYER1 {
    String::from("MCTS(1000) gagne")
} else {
    String::from("Minimax(d=4) gagne")
};
println!("\nResultat : {} en {} coups", result_str, history.len());
println!("Temps moyen MCTS     : {:.3} ms/coup", mean(&times[0]) * 1000.0);
println!("Temps moyen Minimax  : {:.3} ms/coup", mean(&times[1]) * 1000.0);


Coup  1: Rouge joue colonne 2 (27.233 ms)


Coup  2: Jaune joue colonne 3 (0.265 ms)


Coup  3: Rouge joue colonne 3 (24.875 ms)


Coup  4: Jaune joue colonne 3 (0.198 ms)


Coup  5: Rouge joue colonne 3 (16.106 ms)


Coup  6: Jaune joue colonne 3 (0.333 ms)


... (22 outputs left out)

Coup 29: Rouge joue colonne 4 (0.488 ms)


Resultat : MCTS(1000) gagne en 29 coups


Temps moyen MCTS     : 11.704 ms/coup


Temps moyen Minimax  : 0.330 ms/coup


### Lecture du resultat

Les deux camps sont nommes dans les lignes de coup, ce qui permet de suivre
la reponse de chacun. Le temps par coup de MCTS est pilote par le nombre
d'iterations, celui de Minimax par la profondeur : ce sont deux formes de budget
differentes, et c'est leur comparaison a budget de temps comparable qui a un
sens pratique.

## Greedy : l'heuristique sans recherche

Une seule profondeur, mais des regles explicites : gagner si c'est possible,
bloquer si l'adversaire gagne au coup suivant, sinon maximiser l'evaluation
courante en preferant le centre. C'est l'IA qu'on ecrit en quelques lignes ;
dans le tournoi qui suit, elle ne bat que l'IA aleatoire.

In [13]:
/// IA gloutonne : priorite gagner > bloquer > centre > evaluation.
///
/// Regarde un seul coup en avance, sans arbre de recherche.
#[allow(dead_code)]
fn ai_greedy(board: &Board, player: i32, _rng: &mut Mt19937) -> usize {
    let opponent = other(player);
    let moves = board.legal_moves();

    // Priorite 1 : gagner immediatement
    for &col in &moves {
        let mut b = *board;
        b.drop_piece(col, player);
        if b.check_win(player) {
            return col;
        }
    }

    // Priorite 2 : bloquer la victoire adverse
    for &col in &moves {
        let mut b = *board;
        b.drop_piece(col, opponent);
        if b.check_win(opponent) {
            return col;
        }
    }

    // Priorite 3 : preferer le centre, puis l'evaluation statique
    let mut center_order = moves.clone();
    center_order.sort_by_key(|&c| ((c as i32) - (COLS / 2) as i32).abs());
    let mut best_col = center_order[0];
    let mut best_score = i32::MIN;
    for &col in &center_order {
        let mut b = *board;
        b.drop_piece(col, player);
        let score = evaluate_position(&b, player);
        if score > best_score {
            best_score = score;
            best_col = col;
        }
    }

    best_col
}

// Test : Greedy vs Random
let mut rng = Mt19937::new(42);
let (winner, history, times) = play_game(&ai_greedy, &ai_random, &mut rng, false, true);

let result_str = if winner == 0 {
    String::from("Match nul")
} else if winner == PLAYER1 {
    String::from("Greedy gagne")
} else {
    String::from("Random gagne")
};
println!("Resultat : {} en {} coups", result_str, history.len());
println!("Temps moyen Greedy : {:.3} ms/coup", mean(&times[0]) * 1000.0);

Resultat : Greedy gagne en 7 coups


Temps moyen Greedy : 0.009 ms/coup


### Interpretation

Greedy gagne, et beaucoup plus vite que Minimax en temps par coup : evaluer une
position une seule fois coute une fraction de ce que coute un arbre. Sa limite
est structurelle — elle ne voit pas les menaces a deux coups — et c'est
precisement ce que le tournoi qui suit va mesurer.

## Le tournoi : la machinerie

`run_tournament` enchaine les rencontres en round-robin. Chaque paire joue six
parties, dont trois avec le premier participant au trait : c'est ce qui
neutralise l'avantage du premier joueur, sans quoi le classement mesurerait
surtout qui a commence.

Le germe de chaque partie est `g * 1000 + match_idx`, donc fonction du numero de
match et de l'indice de partie, et non d'un etat global : le tournoi est
reproductible quelle que soit la machine.

Le classement attribue trois points par victoire et un par nul, puis departage
les ex aequo par la rapidite.

In [14]:
#[derive(Clone, Copy)]
#[allow(dead_code)]
struct Resultat { victoires: i32, nuls: i32, defaites: i32 }

/// Cle de classement : points (3 par victoire, 1 par nul) puis rapidite.
#[allow(dead_code)]
fn cle_temps(t: &[f64]) -> f64 {
    if t.is_empty() { 0.0 } else { -mean(t) }
}

/// Indices des participants, du premier au dernier au classement.
#[allow(dead_code)]
fn classement(results: &[Resultat], times: &[Vec<f64>]) -> Vec<usize> {
    let mut order: Vec<usize> = (0..results.len()).collect();
    order.sort_by(|&a, &b| {
        let ka = (results[a].victoires * 3 + results[a].nuls, cle_temps(&times[a]));
        let kb = (results[b].victoires * 3 + results[b].nuls, cle_temps(&times[b]));
        kb.partial_cmp(&ka).unwrap_or(std::cmp::Ordering::Equal)
    });
    order
}

/// Tournoi round-robin entre toutes les IA.
///
/// Chaque paire joue `games_per_pair` parties, la moitie avec le premier
/// participant en joueur 1. Le germe de chaque partie depend du numero de
/// match et de l'indice de partie, exactement comme en Python.
///
/// Retourne `(resultats par participant, matrice des confrontations,
/// durees par participant)`.
#[allow(dead_code)]
fn run_tournament(
    participants: &[(&str, &dyn Fn(&Board, i32, &mut Mt19937) -> usize)],
    games_per_pair: usize,
    verbose: bool,
) -> (Vec<Resultat>, Vec<Vec<Option<(i32, i32, i32)>>>, Vec<Vec<f64>>) {
    let n = participants.len();
    let mut results: Vec<Resultat> =
        (0..n).map(|_| Resultat { victoires: 0, nuls: 0, defaites: 0 }).collect();
    let mut matrix: Vec<Vec<Option<(i32, i32, i32)>>> = vec![vec![None; n]; n];
    let mut times: Vec<Vec<f64>> = vec![Vec::new(); n];
    let half = games_per_pair / 2;
    let total_matches = n * (n - 1) / 2;
    let mut match_idx = 0usize;

    if verbose {
        println!(
            "Lancement du tournoi ({} IA, {} parties par paire, {} matchs)...",
            n, games_per_pair, total_matches
        );
        // Estimation ecrite dans la source Python : le `\n` final y est
        // litteral, donc on imprime les deux caracteres, pas un saut de ligne.
        println!("(Estime: ~30-60 secondes)\\n");
    }

    for i in 0..n {
        for j in 0..n {
            if j <= i { continue; }
            match_idx += 1;
            let (name1, ai1) = participants[i];
            let (name2, ai2) = participants[j];
            if verbose {
                print!("  Match {}/{}: {} vs {}... ", match_idx, total_matches, name1, name2);
            }

            let mut w1 = 0i32;
            let mut d = 0i32;
            let mut w2 = 0i32;

            for g in 0..games_per_pair {
                let mut rng = Mt19937::new((g * 1000 + match_idx) as u32);
                let first_is_p1 = g < half;
                let (winner, _hist, t) = if first_is_p1 {
                    play_game(ai1, ai2, &mut rng, false, false)
                } else {
                    play_game(ai2, ai1, &mut rng, false, false)
                };
                let (t1, t2) = if first_is_p1 { (&t[0], &t[1]) } else { (&t[1], &t[0]) };
                times[i].extend(t1.iter().copied());
                times[j].extend(t2.iter().copied());
                if winner == PLAYER1 {
                    if first_is_p1 { w1 += 1; } else { w2 += 1; }
                } else if winner == PLAYER2 {
                    if first_is_p1 { w2 += 1; } else { w1 += 1; }
                } else {
                    d += 1;
                }
            }

            matrix[i][j] = Some((w1, d, w2));
            results[i].victoires += w1;
            results[i].nuls += d;
            results[i].defaites += w2;
            results[j].victoires += w2;
            results[j].nuls += d;
            results[j].defaites += w1;

            if verbose {
                println!("{}W-{}D-{}L", w1, d, w2);
            }
        }
    }

    (results, matrix, times)
}

## Le tournoi : lancement

Cinq IA, six parties par paire, soit dix rencontres. Le resultat de chaque
match s'affiche en `victoires-nuls-defaites` du camp cite en premier.

Minimax en profondeur 6 y joue de vraies parties : cette cellule prend quelques
secondes.

In [15]:
// Definir les participants : les IA sont rangees dans des valeurs possedees,
// d'un type nommable (un `impl Fn` anonyme ne se persiste pas entre cellules).
let participants: Vec<(&str, Box<dyn Fn(&Board, i32, &mut Mt19937) -> usize>)> = vec![
    ("Random", Box::new(ai_random)),
    ("Greedy", Box::new(ai_greedy)),
    ("Minimax(d=4)", Box::new(make_minimax_ai(4))),
    ("Minimax(d=6)", Box::new(make_minimax_ai(6))),
    ("MCTS(500)", Box::new(make_mcts_ai(500))),
];

// La table de references emprunte `participants`. Elle reste locale au bloc :
// une variable de session qui tient un emprunt n'est pas persistable.
let t_start = Instant::now();
let (results, matrix, times) = {
    let refs: Vec<(&str, &dyn Fn(&Board, i32, &mut Mt19937) -> usize)> =
        participants.iter().map(|(name, ai)| (*name, ai.as_ref())).collect();
    run_tournament(&refs, 6, true)
};
let t_total = t_start.elapsed().as_secs_f64();
println!("\nTournoi termine en {:.1}s", t_total);


Lancement du tournoi (5 IA, 6 parties par paire, 10 matchs)...


(Estime: ~30-60 secondes)\n


  Match 1/10: Random vs Greedy... 0W-0D-6L


  Match 2/10: Random vs Minimax(d=4)... 0W-0D-6L


  Match 3/10: Random vs Minimax(d=6)... 0W-0D-6L


  Match 4/10: Random vs MCTS(500)... 0W-0D-6L


... (3 outputs left out)

  Match 8/10: Minimax(d=4) vs Minimax(d=6)... 0W-0D-6L


  Match 9/10: Minimax(d=4) vs MCTS(500)... 6W-0D-0L


  Match 10/10: Minimax(d=6) vs MCTS(500)... 4W-0D-2L


Tournoi termine en 2.4s


### Lecture du resultat

Chaque ligne de match donne le bilan du camp cite en premier, au format
`victoires-nuls-defaites`. Une ligne `0W-0D-6L` signale une domination nette ;
une ligne `4W-0D-2L` signale une opposition reelle. Le classement qui suit
additionne ces bilans : ses colonnes de victoires, de nuls et de defaites
regroupent tous les matchs d'un meme participant.

## Classement et matrice des confrontations

Le classement attribue trois points par victoire et un par nul, puis departage
les ex aequo par le temps moyen par coup — le plus rapide devant. La matrice
donne, ligne par ligne, les resultats du participant cite contre chacun des
autres.

In [16]:
// Deux lectures du meme tournoi : un classement general (une ligne par IA),
// puis les confrontations deux a deux. Les deux lisent les tables remplies
// par l'appel a `run_tournament` de la cellule de code precedente.
println!("\n{}", "=".repeat(80));
println!("  RESULTATS DU TOURNOI");
println!("{}", "=".repeat(80));
println!(
    "{:<18} {:<12} {:<8} {:<12} {:<10} {:<10}",
    "IA", "Victoires", "Nuls", "Defaites", "Score", "ms/coup"
);
println!("{}", "-".repeat(80));

// `classement` rend les indices du premier au dernier ; le score est recalcule
// ici (3 points par victoire, 1 par nul) et le temps moyen par coup, mesure en
// secondes, s'affiche en millisecondes.
for &i in &classement(&results, &times) {
    let r = results[i];
    let score = r.victoires * 3 + r.nuls;
    println!(
        "{:<18} {:<12} {:<8} {:<12} {:<10} {:<10.3}",
        participants[i].0, r.victoires, r.nuls, r.defaites, score, mean(&times[i]) * 1000.0
    );
}
println!("{}", "=".repeat(80));

// `matrix` est triangulaire : `run_tournament` ne remplit que la moitie `i < j`,
// du point de vue du participant d'indice i. Une cellule de la moitie basse se
// lit donc dans la cellule symetrique, en echangeant les deux camps (les
// victoires de l'un sont les defaites de l'autre).
println!("\nMatrice des confrontations (victoires du joueur en ligne) :");
// En-tete : une colonne par participant, de la largeur d'une cellule.
print!("{:<18}", "");
for (name, _) in &participants {
    print!("{:>14}", name);
}
println!();
print!("{:<18}", "");
for _ in &participants {
    print!("{:>14}", "---");
}
println!();

// Une ligne par participant ; la diagonale reste vide : personne ne joue
// contre soi-meme.
for (i, (name, _)) in participants.iter().enumerate() {
    print!("{:>18}", name);
    for j in 0..participants.len() {
        if i == j {
            print!("{:>14}", "---");
        } else if let Some((w_row, draws, w_col)) = matrix[i][j] {
            print!("{:>14}", format!("{}W-{}D-{}L", w_row, draws, w_col));
        } else if let Some((w_col, draws, w_row)) = matrix[j][i] {
            print!("{:>14}", format!("{}W-{}D-{}L", w_row, draws, w_col));
        } else {
            print!("{:>14}", "?");
        }
    }
    println!();
}
let _ = ();


  RESULTATS DU TOURNOI


IA                 Victoires    Nuls     Defaites     Score      ms/coup   


--------------------------------------------------------------------------------


Minimax(d=6)       22           0        2            66         2.790     


Minimax(d=4)       18           0        6            54         0.279     


MCTS(500)          14           0        10           42         5.424     


... (6 outputs left out)

            Greedy      6W-0D-0L           ---      0W-0D-6L      0W-0D-6L      0W-0D-6L


      Minimax(d=4)      6W-0D-0L      6W-0D-0L           ---      0W-0D-6L      6W-0D-0L


      Minimax(d=6)      6W-0D-0L      6W-0D-0L      6W-0D-0L           ---      4W-0D-2L


         MCTS(500)      6W-0D-0L      6W-0D-0L      0W-0D-6L      2W-0D-4L           ---


### Interpretation

Le classement ne mesure pas la force theorique : il mesure la force **a ce
budget de calcul**, sur six parties par paire. Les `---` de la diagonale ne sont
pas des matchs : un participant ne s'affronte pas lui-meme. Un bilan comme
`6W-0D-0L` signale une domination nette ; un partage en trois victoires chacun
signalerait un match equilibre. Les ecarts de temps par coup rappellent que la
colonne « score » ne dit rien du prix paye pour l'obtenir.

## Les memes chiffres, en texte

Le jumeau Python trace trois graphiques : le bilan victoires-nuls-defaites de
chaque IA, les points du classement, et le temps moyen par coup. Le premier est
deja porte par le classement ci-dessus ; les deux autres sont rendus ici sous
forme de lignes de donnees, sans dessin.

In [17]:
// Trois panneaux en Python (bilan, points, temps) ; le bilan est deja dans le
// classement. Les deux autres sont imprimes ici comme donnees, sans dessin.
let ordre = classement(&results, &times);
let mut scores: Vec<String> = Vec::new();
let mut bilans: Vec<String> = Vec::new();
let mut durees: Vec<String> = Vec::new();

for &i in &ordre {
    let r = results[i];
    scores.push(format!("{}={}", participants[i].0, r.victoires * 3 + r.nuls));
    bilans.push(format!(
        "{}={}/{}/{}", participants[i].0, r.victoires, r.nuls, r.defaites
    ));
    durees.push(format!("{}={:.3}", participants[i].0, mean(&times[i]) * 1000.0));
}

println!("Scores (3V + 1N) : {}", scores.join(", "));
println!("Barres empilees (V/N/D) : {}", bilans.join(", "));
println!("Temps moyen par coup (ms) : {}", durees.join(", "));
let _ = ();


Scores (3V + 1N) : Minimax(d=6)=66, Minimax(d=4)=54, MCTS(500)=42, Greedy=18, Random=0


Barres empilees (V/N/D) : Minimax(d=6)=22/0/2, Minimax(d=4)=18/0/6, MCTS(500)=14/0/10, Greedy=6/0/18, Random=0/0/24


Temps moyen par coup (ms) : Minimax(d=6)=2.790, Minimax(d=4)=0.279, MCTS(500)=5.424, Greedy=0.007, Random=0.001


### Lecture du resultat

Les deux series se lisent ensemble : les scores disent qui gagne, les durees
disent a quel prix. Un participant dont le score est bas mais le temps par
coup eleve coute plus qu'il ne rapporte — c'est exactement l'arbitrage qu'un
moteur de jeu doit trancher.

## Le cadre AIMA

AIMA — *Artificial Intelligence: A Modern Approach* — formule un jeu comme un
quadruplet : un etat initial, des actions, un test terminal et une fonction
d'utilite. Le carnet implemente ce cadre une fois, puis branche le Puissance 4
dessus dans la cellule suivante.

En Rust, ce contrat s'ecrit comme un `trait`. Les joueurs sont des fonctions
qui recoivent le jeu, l'etat et le generateur — pas d'objet global.

In [18]:
/// Etat de jeu AIMA : joueur au trait, utilite, plateau, coups possibles.
#[derive(Clone)]
#[allow(dead_code)]
struct GameState {
    to_move: i32,
    utility: i32,
    board: Board,
    moves: Vec<usize>,
}

/// Classe de base pour les jeux a deux joueurs (AIMA chapitre 5).
///
/// En Python, `initial` est un attribut pose par le constructeur ; ici c'est
/// une methode, ce qui laisse l'implementeur libre de l'etat interne.
#[allow(dead_code)]
trait Game {
    fn initial(&self) -> GameState;
    /// Liste des mouvements legaux dans cet etat.
    fn actions(&self, state: &GameState) -> Vec<usize>;
    /// Nouvel etat apres avoir joue ce mouvement.
    fn result(&self, state: &GameState, mv: usize) -> GameState;
    /// True si cet etat est un etat final du jeu.
    fn terminal_test(&self, state: &GameState) -> bool;
    /// Utilite de l'etat pour le joueur (victoire = 1, defaite = -1, nul = 0).
    fn utility(&self, state: &GameState, player: i32) -> f64;
    /// Ordonnancement optionnel des coups : identite par defaut.
    fn order_actions(&self, _state: &GameState, actions: Vec<usize>) -> Vec<usize> {
        actions
    }
}

/// Joue une partie complete avec le framework AIMA.
///
/// Retourne `(gagnant, etat final)` : 1 / -1 / 0.
#[allow(dead_code)]
fn play_game_aima(
    game: &dyn Game,
    player1: &dyn Fn(&dyn Game, &GameState, &mut Mt19937) -> usize,
    player2: &dyn Fn(&dyn Game, &GameState, &mut Mt19937) -> usize,
    rng: &mut Mt19937,
    verbose: bool,
) -> (i32, GameState) {
    let mut state = game.initial();
    let mut move_count = 0;

    while !game.terminal_test(&state) {
        let player = if state.to_move == PLAYER1 { player1 } else { player2 };
        let mv = player(game, &state, rng);
        state = game.result(&state, mv);
        move_count += 1;

        if verbose && move_count <= 50 {
            println!("Coup {}: Joueur {} -> colonne {}", move_count, state.to_move, mv);
        }
    }

    let winner = if state.utility == 1 { 1 } else if state.utility == -1 { -1 } else { 0 };
    (winner, state)
}

/// Joueur Minimax avec elagage alpha-beta, profondeur fixe.
#[allow(dead_code)]
fn alpha_beta_player(game: &dyn Game, state: &GameState, depth: i32, rng: &mut Mt19937) -> usize {
    fn max_value(game: &dyn Game, s: &GameState, mut alpha: f64, beta: f64, d: i32) -> f64 {
        if game.terminal_test(s) || d == 0 {
            return game.utility(s, PLAYER1);
        }
        let mut v = f64::NEG_INFINITY;
        for a in game.actions(s) {
            let child = game.result(s, a);
            let value = min_value(game, &child, alpha, beta, d - 1);
            if value > v { v = value; }
            if v >= beta { return v; }
            if v > alpha { alpha = v; }
        }
        v
    }

    fn min_value(game: &dyn Game, s: &GameState, alpha: f64, mut beta: f64, d: i32) -> f64 {
        if game.terminal_test(s) || d == 0 {
            return game.utility(s, PLAYER1);
        }
        let mut v = f64::INFINITY;
        for a in game.actions(s) {
            let child = game.result(s, a);
            let value = max_value(game, &child, alpha, beta, d - 1);
            if value < v { v = value; }
            if v <= alpha { return v; }
            if v < beta { beta = v; }
        }
        v
    }

    let mut best_score = f64::NEG_INFINITY;
    let mut best_move: Option<usize> = None;
    let mut alpha = f64::NEG_INFINITY;
    let beta = f64::INFINITY;

    for action in game.actions(state) {
        let child = game.result(state, action);
        let score = min_value(game, &child, alpha, beta, depth - 1);
        if score > best_score {
            best_score = score;
            best_move = Some(action);
        }
        if score > alpha { alpha = score; }
    }

    match best_move {
        Some(m) => m,
        None => {
            let moves = game.actions(state);
            if moves.is_empty() { 0 } else { *rng.choice(&moves) }
        }
    }
}

/// Joueur aleatoire (choisit uniformement parmi les coups legaux).
#[allow(dead_code)]
fn random_player(game: &dyn Game, state: &GameState, rng: &mut Mt19937) -> usize {
    let moves = game.actions(state);
    if moves.is_empty() { 0 } else { *rng.choice(&moves) }
}

println!("Framework AIMA de base charge");
println!("  - play_game_aima(), alpha_beta_player(), random_player() disponibles");

Framework AIMA de base charge


  - play_game_aima(), alpha_beta_player(), random_player() disponibles


## L'adaptateur Puissance 4

`ConnectFourAima` implemente le contrat `Game` pour notre plateau. Il ajoute
deux elements venus de l'implementation Java d'AIMA : une analyse des positions
gagnantes, qui note un coup par les menaces qu'il cree et celles qu'il bloque,
et un ordonnancement des actions qui place les meilleurs coups en tete pour
faire couper plus tot.

In [19]:
/// Adaptateur Puissance 4 pour le framework AIMA.
///
/// Enveloppe `Board` avec les extensions inspirees de l'implementation
/// AIMA Java : analyse des positions gagnantes et ordonnancement des coups.
#[allow(dead_code)]
struct ConnectFourAima {
    initial_state: GameState,
}

#[allow(dead_code)]
impl ConnectFourAima {
    fn new() -> ConnectFourAima {
        ConnectFourAima::with_board(Board::new())
    }

    fn with_board(board: Board) -> ConnectFourAima {
        let moves = board.legal_moves();
        ConnectFourAima {
            initial_state: GameState { to_move: PLAYER1, utility: 0, board, moves },
        }
    }

    /// Potentiel d'une action : victoire immediate, menaces creees, menaces
    /// adverses contreessayees, puis bonus de centralite.
    fn analyze_potential_win_positions(&self, state: &GameState, action: usize) -> i32 {
        let player = state.to_move;
        let opponent = other(player);

        // Simuler le coup
        let mut test_board = state.board;
        test_board.drop_piece(action, player);

        // Victoire immediate = score maximum
        if test_board.check_win(player) {
            return 10000;
        }

        // Menaces creees (trois pions et une case vide)
        let threats_created = self.count_threats(&test_board, player);

        // Verifier si on bloque une menace adverse
        let mut test_board_opp = state.board;
        test_board_opp.drop_piece(action, opponent);
        let threats_blocked = if test_board_opp.check_win(opponent) { 1 } else { 0 };

        // Bonus pour le centre (strategiquement plus fort)
        let center_bonus = 7 - ((action as i32) - (COLS / 2) as i32).abs();

        threats_created * 100 + threats_blocked * 500 + center_bonus
    }

    /// Compte les fenetres de quatre cases a trois pions et une case vide.
    fn count_threats(&self, board: &Board, player: i32) -> i32 {
        let mut threats = 0;
        let g = &board.grid;

        for row in 0..ROWS {
            for col in 0..COLS - 3 {
                let window = [g[row][col], g[row][col + 1], g[row][col + 2], g[row][col + 3]];
                if Self::is_threat(&window, player) { threats += 1; }
            }
        }
        for col in 0..COLS {
            for row in 0..ROWS - 3 {
                let window = [g[row][col], g[row + 1][col], g[row + 2][col], g[row + 3][col]];
                if Self::is_threat(&window, player) { threats += 1; }
            }
        }
        for row in 0..ROWS - 3 {
            for col in 0..COLS - 3 {
                let window = [g[row][col], g[row + 1][col + 1], g[row + 2][col + 2], g[row + 3][col + 3]];
                if Self::is_threat(&window, player) { threats += 1; }
            }
        }
        for row in 3..ROWS {
            for col in 0..COLS - 3 {
                let window = [g[row][col], g[row - 1][col + 1], g[row - 2][col + 2], g[row - 3][col + 3]];
                if Self::is_threat(&window, player) { threats += 1; }
            }
        }
        threats
    }

    /// Trois pions du joueur et une case vide dans la fenetre.
    fn is_threat(window: &[i32; CONNECT], player: i32) -> bool {
        let own = window.iter().filter(|&&v| v == player).count();
        let free = window.iter().filter(|&&v| v == EMPTY).count();
        own == 3 && free == 1
    }
}

#[allow(dead_code)]
impl Game for ConnectFourAima {
    fn initial(&self) -> GameState {
        self.initial_state.clone()
    }

    fn actions(&self, state: &GameState) -> Vec<usize> {
        state.moves.clone()
    }

    fn result(&self, state: &GameState, mv: usize) -> GameState {
        let mut board = state.board;
        board.drop_piece(mv, state.to_move);
        let new_moves = board.legal_moves();

        let mut utility = 0;
        if board.check_win(state.to_move) {
            utility = if state.to_move == PLAYER1 { 1 } else { -1 };
        }

        GameState {
            to_move: other(state.to_move),
            utility,
            board,
            moves: new_moves,
        }
    }

    fn terminal_test(&self, state: &GameState) -> bool {
        state.utility != 0 || state.moves.is_empty()
    }

    fn utility(&self, state: &GameState, player: i32) -> f64 {
        if player == PLAYER1 { state.utility as f64 } else { -(state.utility as f64) }
    }

    /// Ordonne les actions par potentiel de victoire decroissant.
    fn order_actions(&self, state: &GameState, actions: Vec<usize>) -> Vec<usize> {
        let mut scored: Vec<(i32, usize)> =
            actions.iter().map(|&a| (self.analyze_potential_win_positions(state, a), a)).collect();
        scored.sort_by(|x, y| y.cmp(x));
        scored.into_iter().map(|(_, a)| a).collect()
    }
}

println!("Adaptateur ConnectFour_AIMA avec analyse des positions gagnantes.");

Adaptateur ConnectFour_AIMA avec analyse des positions gagnantes.


## Approfondissement iteratif

Un alpha-beta a profondeur fixe a un defaut pratique : il peut depasser le
temps disponible, ou en laisser beaucoup inutilise. L'approfondissement
iteratif cherche d'abord a profondeur 1, puis 2, puis 3, en s'arretant des que
le temps est ecoule ; le meilleur coup de la derniere profondeur terminee est
celui qu'on joue. La boucle s'arrete aussi des que la valeur absolue du meilleur
coup depasse le seuil median des utilites : avec des utilites dans `[-1, 1]`, ce
seuil vaut zero, et une note heuristique non nulle suffit donc a conclure.

L'ordonnancement des coups est fait **une fois**, a la racine, par
`order_actions`, et chaque profondeur reutilise cet ordre. C'est un
approfondissement iteratif simplifie : le meilleur coup d'une profondeur n'est
pas remonte en tete de la suivante.

In [20]:
/// Alpha-Beta avec approfondissement iteratif et limite de temps.
///
/// Inspire de `IterativeDeepeningAlphaBetaSearch` d'AIMA Java : arret sur
/// timeout, victoires rapides preferees, ordonnancement des coups. Le jeu est
/// passe a chaque appel en parametre plutot que stocke dans l'objet.
#[allow(dead_code)]
struct IdAb {
    time_limit: f64,
    max_depth: i32,
    util_min: f64,
    util_max: f64,
}

#[allow(dead_code)]
impl IdAb {
    fn new(time_limit: f64) -> IdAb {
        IdAb { time_limit, max_depth: 42, util_min: -1.0, util_max: 1.0 }
    }

    /// Execute l'approfondissement iteratif : `(action, profondeur, valeur)`.
    fn make_decision(
        &self,
        game: &dyn Game,
        state: &GameState,
    ) -> (Option<usize>, i32, f64) {
        let start = Instant::now();
        let mut best_action: Option<usize> = None;
        let mut best_value = f64::NEG_INFINITY;
        let mut depth_reached = 0;
        let player = state.to_move;

        // Ordonner les actions initiales
        let actions = game.order_actions(state, game.actions(state));

        for depth in 1..=self.max_depth {
            if start.elapsed().as_secs_f64() >= self.time_limit {
                break;
            }

            let mut current_best: Option<usize> = None;
            let mut current_value = f64::NEG_INFINITY;

            for &action in &actions {
                let new_state = game.result(state, action);
                let value = self.min_value(
                    game, &new_state, f64::NEG_INFINITY, f64::INFINITY, depth - 1, player, start,
                );

                if let Some(v) = value {
                    if v > current_value {
                        current_value = v;
                        current_best = Some(action);
                    }
                }
            }

            if current_best.is_some() {
                best_action = current_best;
                best_value = current_value;
                depth_reached = depth;
            }

            // Arreter si victoire certaine trouvee
            if self.has_safe_winner(best_value) {
                break;
            }
        }

        (best_action, depth_reached, best_value)
    }

    fn max_value(
        &self,
        game: &dyn Game,
        state: &GameState,
        mut alpha: f64,
        beta: f64,
        depth: i32,
        player: i32,
        start: Instant,
    ) -> Option<f64> {
        if start.elapsed().as_secs_f64() >= self.time_limit {
            return None;
        }
        if game.terminal_test(state) {
            return Some(self.eval_terminal(game, state, player));
        }
        if depth == 0 {
            return Some(self.eval(state, player));
        }

        let mut v = f64::NEG_INFINITY;
        for action in game.order_actions(state, game.actions(state)) {
            let new_state = game.result(state, action);
            let value = self.min_value(game, &new_state, alpha, beta, depth - 1, player, start);
            let value = match value {
                Some(x) => x,
                None => return None,
            };
            if value > v { v = value; }
            if v >= beta {
                return Some(v);
            }
            if v > alpha { alpha = v; }
        }
        Some(v)
    }

    fn min_value(
        &self,
        game: &dyn Game,
        state: &GameState,
        alpha: f64,
        mut beta: f64,
        depth: i32,
        player: i32,
        start: Instant,
    ) -> Option<f64> {
        if start.elapsed().as_secs_f64() >= self.time_limit {
            return None;
        }
        if game.terminal_test(state) {
            return Some(self.eval_terminal(game, state, player));
        }
        if depth == 0 {
            return Some(self.eval(state, player));
        }

        let mut v = f64::INFINITY;
        for action in game.order_actions(state, game.actions(state)) {
            let new_state = game.result(state, action);
            let value = self.max_value(game, &new_state, alpha, beta, depth - 1, player, start);
            let value = match value {
                Some(x) => x,
                None => return None,
            };
            if value < v { v = value; }
            if v <= alpha {
                return Some(v);
            }
            if v < beta { beta = v; }
        }
        Some(v)
    }

    /// Evaluation d'un etat terminal, ajustee pour preferer les victoires rapides.
    fn eval_terminal(&self, game: &dyn Game, state: &GameState, player: i32) -> f64 {
        let utility = game.utility(state, player);
        if self.has_safe_winner(utility) {
            let moves = state.board.move_count() as f64;
            if utility > 0.0 {
                utility - moves / 1000.0
            } else {
                utility + moves / 1000.0
            }
        } else {
            utility
        }
    }

    /// Evaluation d'un etat non terminal (heuristique de position).
    fn eval(&self, state: &GameState, player: i32) -> f64 {
        evaluate_position(&state.board, player) as f64 / 100.0
    }

    /// Une valeur au-dela du seuil median indique une victoire certaine.
    fn has_safe_winner(&self, value: f64) -> bool {
        let threshold = (self.util_min + self.util_max) / 2.0;
        value.abs() > threshold
    }
}

/// Cree un joueur utilisant l'approfondissement iteratif.
#[allow(dead_code)]
fn make_id_alphabeta_player(
    time_limit: f64,
) -> impl Fn(&dyn Game, &GameState, &mut Mt19937) -> usize {
    let search = IdAb::new(time_limit);
    move |game: &dyn Game, state: &GameState, _rng: &mut Mt19937| -> usize {
        match search.make_decision(game, state).0 {
            Some(action) => action,
            None => 0,
        }
    }
}

println!("IterativeDeepeningAlphaBeta charge.");

IterativeDeepeningAlphaBeta charge.


## Benchmark des deux approches

Deux alpha-beta, deux budgets : l'un a profondeur fixe 4, l'autre avec
approfondissement iteratif et une demi-seconde par coup. Tous deux affrontent
la meme IA aleatoire, avec des graines distinctes.

In [21]:
/// Compare les approches AIMA sur plusieurs parties contre un joueur aleatoire.
///
/// Affiche le nombre de victoires et le temps moyen par partie de chaque
/// approche. Le dictionnaire de resultats de Python n'est pas reproduit : son
/// affichage n'etait qu'un echo d'objet, sans valeur pedagogique.
#[allow(dead_code)]
fn benchmark_aima_approaches(n_games: usize) {
    println!("Benchmark des approches AIMA");
    println!("{}", "=".repeat(60));

    // Partie 1 : AB-classic (profondeur 4, sans ordonnancement) vs Random
    let mut ab_wins = 0i32;
    let mut ab_times: Vec<f64> = Vec::new();
    for i in 0..n_games {
        let mut rng = Mt19937::new((42 + i) as u32);
        let game = ConnectFourAima::new();

        let start = Instant::now();
        let ab_player =
            |g: &dyn Game, s: &GameState, r: &mut Mt19937| alpha_beta_player(g, s, 4, r);
        let (result, _) = play_game_aima(&game, &ab_player, &random_player, &mut rng, false);
        ab_times.push(start.elapsed().as_secs_f64());

        if result == 1 {
            ab_wins += 1;
        }
    }

    println!("AB-classic (d=4): {}/{} victoires", ab_wins, n_games);

    // Partie 2 : ID-AB (limite 0.5 s) vs Random
    let mut id_wins = 0i32;
    let mut id_times: Vec<f64> = Vec::new();
    for i in 0..n_games {
        let mut rng = Mt19937::new((100 + i) as u32);
        let game = ConnectFourAima::new();
        let search = IdAb::new(0.5);

        let start = Instant::now();
        let id_player = |g: &dyn Game, s: &GameState, _r: &mut Mt19937| {
            match search.make_decision(g, s).0 {
                Some(action) => action,
                None => 0,
            }
        };
        let (result, _) = play_game_aima(&game, &id_player, &random_player, &mut rng, false);
        id_times.push(start.elapsed().as_secs_f64());

        if result == 1 {
            id_wins += 1;
        }
    }

    println!("ID-AB (0.5s):     {}/{} victoires", id_wins, n_games);

    // Temps moyen par partie
    println!("\nTemps moyen par partie:");
    println!("  AB-classic: {:.3} ms", mean(&ab_times) * 1000.0);
    println!("  ID-AB:      {:.3} ms", mean(&id_times) * 1000.0);
}

println!("Test des techniques avancees inspirees d'AIMA Java\n");
benchmark_aima_approaches(3);

Test des techniques avancees inspirees d'AIMA Java


Benchmark des approches AIMA


AB-classic (d=4): 3/3 victoires


ID-AB (0.5s):     3/3 victoires


Temps moyen par partie:


  AB-classic: 0.337 ms


  ID-AB:      0.049 ms


### Interpretation

Le nombre de victoires dit si l'approche tient sa promesse face a un joueur
faible — c'est le minimum attendu, et les deux l'atteignent. Les temps appellent
une remarque : l'approfondissement iteratif porte un budget de demi-seconde par
coup, mais son arret sur valeur sure le fait conclure des le premier niveau, des
que la meilleure note n'est pas exactement nulle. Il ne depense donc presque pas
son budget : le temps affiche mesure cet arret premature, pas la force de la
methode.

## Exercice 1 : negamax

Le negamax exprime le minimax sans alterner deux fonctions : le score d'un
noeud est toujours evalue du point de vue du joueur au trait, et l'adversaire
retourne simplement le signe. Ecrire `negamax` puis verifier qu'il choisit la
meme colonne que Minimax sur une position donnee est le meilleur test de
comprehension de l'algorithme.

In [22]:
/// Exercice : negamax.
///
/// TODO etudiant : implementer negamax, puis verifier qu'il choisit la meme
/// colonne que Minimax sur la position de test.
#[allow(dead_code)]
fn negamax(
    _board: &mut Board,
    _depth: i32,
    _alpha: f64,
    _beta: f64,
    _player: i32,
) -> Option<(usize, f64)> {
    // TODO etudiant : retourner (colonne, score) du meilleur coup.
    None
}

// Verification : Negamax vs Minimax (devraient jouer pareil)
let mut test_board = Board::new();
test_board.drop_piece(3, PLAYER1);
test_board.drop_piece(4, PLAYER2);
test_board.drop_piece(3, PLAYER1);

match negamax(&mut test_board, 4, f64::NEG_INFINITY, f64::INFINITY, PLAYER2) {
    Some((col_neg, _score)) => {
        let ai_mm4 = make_minimax_ai(4);
        let mut rng_probe = Mt19937::new(0);
        let col_mm = ai_mm4(&test_board, PLAYER2, &mut rng_probe);
        println!("Minimax choisit colonne : {}", col_mm);
        println!("Negamax choisit colonne : {}", col_neg);
        println!(
            "Meme decision : {}",
            if col_mm == col_neg { "Oui" } else { "Non (normal si scores egaux)" }
        );
    }
    None => {
        println!("Exercice a completer : implementez negamax() pour comparer avec Minimax.");
        println!("Attendu : negamax doit retourner (colonne, score), meme decision que Minimax.");
    }
}
let _ = ();


Exercice a completer : implementez negamax() pour comparer avec Minimax.


Attendu : negamax doit retourner (colonne, score), meme decision que Minimax.


## Exercice 2 : approfondissement iteratif maison

`IdAb` existe dans la section AIMA ; l'exercice demande de le refaire sans le
cadre, en s'appuyant directement sur le plateau. Le test fait jouer
l'implementation contre l'IA aleatoire et mesure son temps par coup.

In [23]:
/// Exercice : approfondissement iteratif maison.
///
/// TODO etudiant : implementer une recherche a profondeur croissante bornee
/// par une limite de temps, puis la faire jouer contre l'IA aleatoire.
#[allow(dead_code)]
fn iterative_deepening_minimax(
    _board: &Board,
    _player: i32,
    _time_limit: f64,
    _rng: &mut Mt19937,
) -> Option<usize> {
    // TODO etudiant : retourner la meilleure colonne trouvee.
    None
}

// Test
let mut rng_probe = Mt19937::new(42);
let probe = iterative_deepening_minimax(&Board::new(), PLAYER1, 0.5, &mut rng_probe);
match probe {
    Some(_) => {
        let ai_id = |board: &Board, player: i32, rng: &mut Mt19937| -> usize {
            iterative_deepening_minimax(board, player, 0.5, rng).unwrap_or(0)
        };
        let mut rng = Mt19937::new(42);
        let (winner, history, times) = play_game(&ai_id, &ai_random, &mut rng, false, false);
        let result_str = if winner == 0 {
            "Match nul"
        } else if winner == PLAYER1 {
            "ID gagne"
        } else {
            "Random gagne"
        };
        println!("ID(0.5s) vs Random : {} en {} coups", result_str, history.len());
        println!("Temps moyen ID : {:.3} ms/coup", mean(&times[0]) * 1000.0);
    }
    None => {
        println!("Exercice a completer : implementez iterative_deepening_minimax() pour tester.");
        println!("Attendu : ID(0.5s) devrait battre Random (>80% de victoires).");
    }
}
let _ = ();


Exercice a completer : implementez iterative_deepening_minimax() pour tester.


Attendu : ID(0.5s) devrait battre Random (>80% de victoires).


## Exercice 3 : ordonnancement des coups et comptage de noeuds

L'elagage alpha-beta est d'autant plus efficace que les bons coups sont essayes
tot. L'exercice demande deux fonctions : `order_moves`, qui trie les coups
legaux par proximite du centre, et `count_nodes_minimax`, qui compte les noeuds
explores avec et sans ordonnancement. Le rapport des deux comptes est le
speedup — la mesure directe de ce que l'ordonnancement rapporte.
**Ce que Rust apporte, à profondeur égale** : le même protocole (Minimax alpha-beta contre l'IA aléatoire, dix parties par profondeur, cellule 27 du jumeau Python / cellule 25 ici) mesure d=2 à 23,9 ms par coup côté Python contre 0,022 ms ici, d=4 à 257,3 ms contre 0,313 ms et d=6 à 2 823,9 ms contre 3,644 ms — un facteur d'environ ×1 100 à ×775 selon la profondeur. Les deux chiffres viennent des sorties committées des deux carnets, mesurées sur des machines différentes : c'est un ordre de grandeur, pas un banc croisé. Ce que Rust ne change pas : les dix victoires sur dix à chaque profondeur et la courbe de coût de la recherche — la profondeur 6 coûte à Rust autant de fois la profondeur 2 qu'à Python (×166 contre ×118, même élagage, même croissance).

In [24]:
/// Exercice : ordonnancement des coups et comptage de noeuds.
///
/// TODO etudiant : implementer order_moves (trier les coups par proximite du
/// centre) puis count_nodes_minimax (compter les noeuds explores).
#[allow(dead_code)]
fn order_moves(_board: &Board, actions: Vec<usize>) -> Vec<usize> {
    // TODO etudiant : retourner les actions triees.
    actions
}

/// TODO etudiant : retourner le nombre de noeuds explores.
#[allow(dead_code)]
fn count_nodes_minimax(
    _board: &Board,
    _depth: i32,
    _player: i32,
    _use_ordering: bool,
) -> Option<usize> {
    // TODO etudiant : compter les appels recursifs.
    None
}

// Test sur une position de milieu de partie
let mut test_board = Board::new();
for (col, pl) in [(3usize, 1), (3, 2), (4, 1), (2, 2), (5, 1), (1, 2), (2, 1), (4, 2)] {
    test_board.drop_piece(col, pl);
}

let mut mesure = false;
for depth in [4, 5, 6] {
    if let (Some(nodes_basic), Some(nodes_smart)) = (
        count_nodes_minimax(&test_board, depth, PLAYER1, false),
        count_nodes_minimax(&test_board, depth, PLAYER1, true),
    ) {
        let speedup = if nodes_smart > 0 {
            nodes_basic as f64 / nodes_smart as f64
        } else {
            f64::INFINITY
        };
        println!(
            "Profondeur {}: sans ordering = {:>8} noeuds, avec ordering = {:>8} noeuds, speedup = {:.2}x",
            depth, nodes_basic, nodes_smart, speedup
        );
        mesure = true;
    }
}
if !mesure {
    println!("Exercice a completer : implementez order_moves() et count_nodes_minimax().");
    println!("Attendu : speedup 2-5x avec ordering (moins de noeuds explores).");
}
let _ = ();


Exercice a completer : implementez order_moves() et count_nodes_minimax().


Attendu : speedup 2-5x avec ordering (moins de noeuds explores).


## Ce que le carnet a montre

Le meme jeu, six IA et un tournoi tiennent en quelques centaines de lignes de
Rust sans aucune dependance externe. Les quatre familles d'approche se
distinguent nettement :

| Famille | Principe | Cout | Faiblesse |
|---|---|---|---|
| Aleatoire | tir uniforme | negligeable | ne construit rien |
| Greedy | une evaluation, quelques regles | une evaluation par coup | ne voit pas deux coups plus loin |
| Minimax alpha-beta | arbre avec elagage | exponentiel en profondeur | depend de la fonction d'evaluation |
| MCTS | simulations + UCB1 | lineaire en iterations | bruit statistique, pas de garantie |

Le cadre AIMA montre que ces algorithmes partagent une meme structure : un
etat, des actions, un test terminal, une utilite. Une fois ce contrat pose,
changer d'algorithme ne change plus que la maniere d'explorer.

Deux techniques reviennent dans tous les moteurs de jeu reels : l'**ordre des
coups**, qui fait couper plus tot, et le **budget de temps adaptatif**, qui
depense le calcul la ou il rapporte. Les exercices proposent de les mesurer
soi-meme plutot que de les croire sur parole.